# grc-joint LAS experiment — comma-fix + relation-on-predicted-arcs

Tests two bundled changes against the shipped recipe, on **5 seeds**:

1. **Converter comma fix** — non-coordination commas attach to the following token (the UD-Perseus convention). Offline-measured: +0.65 head+label converter agreement.
2. **Relation-on-predicted-arcs** — the relation head is also supervised at the model's *predicted* heads, closing the train/test mismatch.

Judged by **seed-mean** vs. the original recipe's seed-mean (Perseus LAS **83.97 ± 0.43**, UAS 88.75 ± 0.33) and best-published (LAS 83.98, UAS 88.20). A robust win = new seed-mean clears the bar, not a single lucky seed.

**No commits to main** — the notebook clones main and applies the changes in-place. **Runtime** (G4, RTX PRO 6000 Blackwell): ~12-15 min/seed, ~1-1.5 h for 5. Send back `exp_results.zip` (or `summary.json`).

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "| bf16", torch.cuda.is_bf16_supported())

In [ ]:
%cd /content
!rm -rf pyaegean
!git clone --depth 1 https://github.com/ryanpavlicek/pyaegean.git
%cd /content/pyaegean
!pip install -q .
!pip install -q transformers
import importlib, aegean; importlib.invalidate_caches()
print("aegean", aegean.__version__)

In [ ]:
# Apply the experiment to the freshly-cloned main (no commits): comma-forward converter
# fix + relation-on-predicted-arcs. Assert-guarded so a missed anchor fails loudly.
import pathlib
EDITS = [
    ('training/agdt_ud_deps.py', '            move_children(i, promoted, keep={i, promoted})\n\n    # AuxK (final punctuation) hangs off the root token; force a single root', '            move_children(i, promoted, keep={i, promoted})\n\n    # non-coordination commas (AuxX) attach to the immediately following token — the\n    # UD-Perseus convention (coordination separators are handled above). Guard against a\n    # cycle when that next token is the comma\'s own dependent.\n    for i in range(n):\n        if _base(rels[i]) != "AuxX":\n            continue\n        tgt = i + 1 if i + 1 < n else i - 1\n        if 0 <= tgt != i:\n            j, hops = tgt, 0\n            while j != -1 and hops <= n:\n                if j == i:\n                    break\n                j, hops = out_head[j], hops + 1\n            else:\n                out_head[i] = tgt\n        fixed[i] = "punct"\n\n    # AuxK (final punctuation) hangs off the root token; force a single root'),
    ('training/train_full.py', '                loss = loss + ce(rel_at_gold.flatten(0, 1), gr.flatten())\n                loss = loss + ce(lem.flatten(0, 1), gs.flatten())', "                loss = loss + ce(rel_at_gold.flatten(0, 1), gr.flatten())\n                # also supervise relations at the model's PREDICTED heads, not only gold\n                # arcs: at inference the relation is read at the predicted head, so this\n                # closes the train/test mismatch (gold heads -> argmax heads).\n                ph_pred = arc.argmax(dim=2)\n                rel_at_pred = rel.permute(0, 2, 3, 1).gather(\n                    2, ph_pred.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 1, rel.size(1))\n                ).squeeze(2)\n                loss = loss + ce(rel_at_pred.flatten(0, 1), gr.flatten())\n                loss = loss + ce(lem.flatten(0, 1), gs.flatten())"),
]
for p, o, n in EDITS:
    t = pathlib.Path(p).read_text(encoding='utf-8')
    assert t.count(o) == 1, p + ': anchor found ' + str(t.count(o)) + 'x (expected 1)'
    pathlib.Path(p).write_text(t.replace(o, n), encoding='utf-8')
    print('patched', p)
assert 'rel_at_pred' in pathlib.Path('training/train_full.py').read_text(encoding='utf-8')
assert 'non-coordination commas' in pathlib.Path('training/agdt_ud_deps.py').read_text(encoding='utf-8')
print('OK: comma fix + relation-on-predicted-arcs applied')

In [ ]:
!python training/build_full_dataset.py --with-extras
import os
assert os.path.exists("training/data/full-train.jsonl"), "dataset build failed"
print("dataset ready (built with the comma-fixed converter)")

In [ ]:
# Smoke test: 1 epoch on 500 sentences confirms the patched loss trains with no shape error
!python training/train_full.py --limit-train 500 --epochs 1 --seed 0 --out /content/smoke
import os
assert os.path.exists("/content/smoke/model"), "smoke train produced no checkpoint"
print("smoke OK -> patched training runs end to end")

In [ ]:
SEEDS = [1, 2, 3, 4, 5]   # reduce to [1, 2, 3] if G4 time is tight
EPOCHS = 12
COMPOSE = "lookup-first"  # lemma composition only; irrelevant to LAS/UAS
import os
os.makedirs("/content/exp_results", exist_ok=True)

In [ ]:
import subprocess, json, time, os, sys
from pathlib import Path

ENV = {**os.environ, "PYTHONUNBUFFERED": "1"}  # force the child to flush so output streams live

def run(cmd, tag):
    print(f"   $ {tag}  ->  live output:", flush=True)
    t = time.time()
    proc = subprocess.Popen(cmd, cwd="/content/pyaegean", env=ENV,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:                      # stream each line as it arrives
        sys.stdout.write("   | " + line); sys.stdout.flush()
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"{tag} FAILED: {' '.join(cmd)}")
    print(f"   [{tag} done in {time.time()-t:.0f}s]", flush=True)

t0 = time.time()
for i, s in enumerate(SEEDS, 1):
    print(f"\n{'='*16} SEED {s}  ({i}/{len(SEEDS)}) {'='*16}   [total elapsed {(time.time()-t0)/60:.0f} min]", flush=True)
    out = f"/content/pyaegean/training/out/exp-seed{s}"
    res = Path(f"/content/exp_results/seed{s}"); res.mkdir(parents=True, exist_ok=True)
    run(["python", "training/train_full.py", "--model", "bowphs/GreBerta", "--epochs", str(EPOCHS),
         "--lr", "3e-5", "--batch", "32", "--max-len", "256", "--seed", str(s), "--out", out], f"train seed {s} (12 epochs)")
    assert Path(f"{out}/model").exists()
    for tb in ("perseus", "proiel"):
        run(["python", "training/eval_full_ud.py", "--checkpoint", f"{out}/model", "--compose", COMPOSE,
             "--treebank", tb, "--split", "test", "--out", str(res / f"ud-{tb}-test.json")], f"eval {tb}")
    p = json.loads((res / "ud-perseus-test.json").read_text())
    print(f">>> SEED {s} COMPLETE: Perseus LAS {p['las']*100:.2f}  UAS {p['uas']*100:.2f}  lemma {p['lemma']*100:.2f}   "
          f"[{i}/{len(SEEDS)} seeds done, {(time.time()-t0)/60:.0f} min elapsed]", flush=True)

In [ ]:
import json, statistics as st
from pathlib import Path
BASE = {"upos": 96.87, "xpos": 93.43, "ufeats": 96.11, "lemma": 94.34, "uas": 88.75, "las": 83.97}  # original recipe, 5-seed mean
BEST = {"uas": 88.20, "las": 83.98}  # best published 2024
mets = ("upos", "xpos", "ufeats", "lemma", "uas", "las")
summ = {}
print("PERSEUS test  (experiment vs original-recipe seed-mean):")
for m in mets:
    xs = [json.loads(Path(f"/content/exp_results/seed{s}/ud-perseus-test.json").read_text())[m] * 100
          for s in SEEDS if Path(f"/content/exp_results/seed{s}/ud-perseus-test.json").exists()]
    mean = st.mean(xs); sd = st.pstdev(xs) if len(xs) > 1 else 0.0
    summ[m] = {"mean": mean, "std": sd, "runs": xs}
    print(f"  {m:7s} {mean:6.2f} +/- {sd:.2f}   baseline {BASE[m]:6.2f}   delta {mean-BASE[m]:+.2f}")
la = summ["las"]
print(f"\nLAS {la['mean']:.2f} +/- {la['std']:.2f}  (baseline 83.97, best-pub 83.98)")
print(f"  robustly clears best-pub (mean - 2sigma > 83.98)? {la['mean'] - 2*la['std'] > 83.98}")
pro = [json.loads(Path(f"/content/exp_results/seed{s}/ud-proiel-test.json").read_text())["las"] * 100
       for s in SEEDS if Path(f"/content/exp_results/seed{s}/ud-proiel-test.json").exists()]
print(f"\nPROIEL LAS (OOD guard): {st.mean(pro):.2f} +/- {st.pstdev(pro) if len(pro)>1 else 0:.2f}  (baseline ~63.5; must not regress)")
Path("/content/exp_results/summary.json").write_text(json.dumps(summ, indent=2))
print("\nwrote summary.json")

In [ ]:
import shutil
shutil.make_archive("/content/exp_results", "zip", "/content/exp_results")
try:
    from google.colab import files
    files.download("/content/exp_results.zip")
except Exception as e:
    print("grab /content/exp_results.zip from the file browser (", e, ")")